# `Industrial Machine Learning on Hadoop and Spark`
## `Seminar 11: Spark Streaming + Kafka`

### `Nakhodnov Maksim`
#### `Bremen, 2025`

What you will learn from this notebook:

* Streaming Data Processing
    * Kafka
    * Delta Lake

In [ ]:
%pip install pyspark pyarrow sparknlp -q

In [1]:
import os
import glob
from typing import Iterator

import pandas as pd

import matplotlib.pyplot as plt
from IPython.display import display

import matplotlib_inline

matplotlib_inline.backend_inline.set_matplotlib_formats('pdf', 'svg')

In [2]:
import pyspark.sql.types as T
import pyspark.sql.functions as F
import pyspark.sql.streaming.state
import pyspark.sql.streaming.listener

from pyspark.sql import SparkSession
from pyspark import SparkConf, SparkContext
from pyspark.streaming import StreamingContext

conf = (
    SparkConf()
        .set('spark.ui.port', '4050')
        .set('spark.driver.memory', '15g')
        # The number of partitions should be set to the number of cores in local mode
        # By default it is set to 200, which is not good for local development
        # Critical for performance on small data/local mode to avoid scheduling overhead
        .set("spark.sql.shuffle.partitions", "12")
        # RocksDB
        .set(
            "spark.sql.streaming.stateStore.providerClass", 
            "org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider"
        )
        # Kafka + Delta Lake
        .set('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.0,io.delta:delta-spark_2.12:3.3.2')
        .set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .setMaster('local[*]')
)
sc = SparkContext(conf=conf)
spark = SparkSession(sc)

25/12/10 22:46:51 WARN Utils: Your hostname, ASUS-UM5606W resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/12/10 22:46:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/maksim64/miniconda3/envs/spark/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/maksim64/.ivy2/cache
The jars for the packages stored in: /home/maksim64/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-741ed9be-4012-41ff-9c8f-ae8e4fdc45a5;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.2.0 in central
	found org.apache.kafka#kafka-clients;2.8.0 in central
	found org.lz4#lz4-java;1.7.1 in central
	found org.xerial.snappy#snappy-java;1.1.8.4 in central
	found org.slf4j#slf4j-api;1.7.30 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.1 in central
	found org.spark-project.spark#unused;1.0.0 in central
	found org.apache.hadoop#hadoop-client-api;3.3.1 in central
	found org.apache.htrace#htrace-core4;4.1.0-incubating in central
	found commons-logging#commons-logging;1.1.3 in central
	fo

## `The Real-Time Data Stack`

One of the most popular Spark Streaming use cases is reading data from Kafka and writing to Delta Lake format.

Kafka is a message broker that allows sending and receiving messages between applications. The main task of Kafka is sharded, fault-tolerant processing of large amounts of data in real-time.

Delta Lake is an open format for storing data that adds reliability, performance, and transaction management features to existing data lakes:
* adds an optimized storage layer in table format with ACID transaction support
* enables scalable metadata processing
* allows updating data in analytical tables stored as Parquet files on HDFS or S3-compatible storage
* allows processing batch queries and performing streaming data operations

### `Kafka`

![](https://cdn.prod.website-files.com/62e7bb0016ec360b2482a37e/6311e5cf2e256f1cb936f826_batchprocessing_with_apachekafka_apachekafka-1024x317.jpeg)

Apache Kafka is a distributed event streaming platform. Unlike traditional message queues (like RabbitMQ) that delete messages once consumed, **Kafka functions as a distributed log**. It persists messages on disk for a configurable retention period, allowing multiple consumers to read the same data at different speeds and different times.

1. **Topic**: A category or feed name to which records are published. Think of it as a folder in a filesystem or a table in a database.
2. **Partition**: To scale beyond a single server, topics are split into partitions.
3. **Parallelism**: A topic with 10 partitions allows 10 consumers to read simultaneously.
4. **Ordering**: Kafka guarantees order only within a partition, not across the entire topic.
5. **Offset**: Each message within a partition gets a unique ID called an offset. It acts as a bookmark. Spark Streaming tracks these offsets to know exactly what data it has processed.
6. **Broker**: A single Kafka server. A cluster is made of multiple brokers.
7. **Replication**: Partitions are replicated across multiple brokers (usually 3) for fault tolerance. If one broker dies, another takes over with zero data loss.

![](https://seifrajhi.github.io/static/925e848d9930f34df57dc09f148f80c4/b5a09/architecture.png)

**Why Kafka for Spark Streaming?**
* **Decoupling**: Producers (web apps, IoT devices) don't need to know about Consumers (Spark, Flink). They just send data to Kafka.
* **Buffer**: If Spark goes down for maintenance, Kafka holds the data. When Spark restarts, it picks up exactly where it left off.
* **Replayability**: You can "rewind" your Spark application to re-process data from last week if you find a bug in your code.

### `Delta Lake`

Standard Data Lakes (built on HDFS or S3 with Parquet files) suffer from major reliability issues:
* **Partial Writes**: If a job fails midway, you get half-written files.
* **Dirty Reads**: If you try to read while someone is writing, the job crashes.
* **Update Nightmare**: You cannot update a single row in a CSV/Parquet file; you must rewrite the whole file.

Delta Lake solves this by adding a Transactional Layer on top of your existing storage.

![](https://delta-io.github.io/delta-rs/how-delta-lake-works/contents-of-delta-table.png)


**How Delta Lake Works:**

It stores data in standard Parquet files but adds a Delta Log (a _delta_log folder). This log records every single transaction (ACID):
* **Add File**: "I just added part-001.parquet."
* **Remove File**: "I logically deleted part-002.parquet."
* **Metadata**: "The schema is now {col1: int, col2: string}."


**Key Features for Streaming:**
* **ACID Transactions**: Spark ensures that a batch of data is either fully written or not written at all.
* **Schema Enforcement**: If the incoming Kafka data has a wrong column type, Delta blocks the write.
* **Upserts (Merge)**: Unlike standard Parquet, Delta supports SQL MERGE operations. This allows us to handle Deduplication.
* **Time Travel**: Because Delta keeps old Parquet files (until you run VACUUM), you can query an older version of the table.

| Feature | Delta Lake (OLAP) | Generic SQL DB (OLTP) |
| :--- | :--- | :--- |
| **Data Volume** | **Petabytes (PB)** (unlimited scale) | **Gigabytes/Terabytes** (typically <10TB) |
| **Write Speed** | **High Throughput** (1M+ rows/sec ingestion) | **Low Latency** (50k+ concurrent transactions/sec) |
| **Query Latency** | **Seconds to Minutes** (Scans billions of rows) | **< 10 Milliseconds** (Point lookups by ID) |
| **Update Cost** | **Heavy** (Must rewrite full ~128MB Parquet files) | **Light** (Updates specific ~8KB memory pages) |
| **Storage Cost** | **~$23 / TB / Month** (S3/Blob Storage) | **~$1,200+ / TB / Month** (Provisioned IOPS SSD) |
| **Scaling Limit** | **1,000+ Nodes** (Horizontal / Distributed) | **~128 Cores / 4TB RAM** (Vertical / Single Node) |
| | | |
| | | |
| **Goal** | Analyze the business | Run the business |
| **Typical Query** | Complex, returns aggregates (e.g., "Sum of all sales in Q4") | Simple, returns few records (e.g., "Find Order #101") |
| **Data Source** | Data Warehouse (fed by multiple OLTP sources) | The application itself (Operational DB) |
| **Volumetrics** | Fewer transactions, but huge data volume | Millions of small transactions |

### `The "Exactly-Once" semantics`

The combination of Kafka + Spark + Delta provides End-to-End Exactly-Once Semantics.

* **Read**: Spark reads a batch from Kafka and tracks the offsets.
* **Process**: Spark performs transformations.
* **Write (Checkpoint)**: Before committing to Delta, Spark saves the Kafka offsets to a Checkpoint Location (Write-Ahead Log).
* **Commit**: Spark commits the data to the Delta Log.
* **Failure Recovery**: If the driver crashes, Spark restarts, reads the checkpoint, sees it successfully processed offsets 100-200, and immediately requests offset 201 from Kafka. No data is lost, and no data is processed twice.

### `The Medallion Architecture`

A data design pattern used in Delta Lake to incrementally improve data quality as it flows through the pipeline.

* **The Bronze Layer (Raw)** acts as the landing zone, ingesting data exactly as it arrives from sources like Kafka.
* **The Silver Layer (Refined)** represents validated, cleansed, and enriched data. 
* **The Gold Layer (Aggregated)** contains highly processed data.

![](https://www.databricks.com/sites/default/files/inline-images/building-data-pipelines-with-delta-lake-120823.png)

### `Example`

Before we can process data in Spark, we need an active source generating events. Under the hood, the script manages the connection to the Kafka Broker, ensures the target topic exists, and asynchronously dispatches the data as raw bytes.

In [ ]:
%pip install confluent-kafka rich

In [3]:
from rich.console import Console
from rich.syntax import Syntax

syntax = Syntax(open("./producer.py").read(), "python", line_numbers=True)
console = Console()
console.print(syntax)

    1 import sys                                                                                                   
    2 import json                                                                                                  
    3 import time                                                                                                  
    4 import random                                                                                                
    5                                                                                                              
    6 from confluent_kafka import Producer                                                                         
    7 from confluent_kafka.admin import AdminClient, NewTopic                                                      
    8                                                                                                              
    9 # Configuration                                                                                              
   10 BOOTSTRAP_SERVERS = 'localhost:9092'                                                                         
   11 TOPIC_NAME = 'words-topic-spark'                                                                             
   12                                                                                                              
   13 SENTENCES = [                                                                                                
   14     "spark streaming is powerful",                                                                           
   15     "kafka provides real-time data",                                                                         
   16     "delta lake ensures acid transactions",                                                                  
   17     "python is great for data engineering",                                                                  
   18     "big data requires distributed computing",                                                               
   19     "apache spark handles batch and streaming",                                                              
   20     "data lakehouse architecture is modern",                                                                 
   21     "learning hadoop and spark is fun"                                                                       
   22 ]                                                                                                            
   23                                                                                                              
   24                                                                                                              
   25 def print_cluster_metadata():                                                                                
   26     """                                                                                                      
   27     Demonstrates the 'Discovery' mechanism.                                                                  
   28     We connect to the bootstrap server, but we ask for the full Cluster Metadata.                            
   29     This prints ALL brokers in the cluster, proving we don't need to hardcode them.                          
   30     """                                                                                                      
   31     print(f"🔍 Connecting to Bootstrap Server: {BOOTSTRAP_SERVERS}...")                                      
   32                                                                                                              
   33     # The AdminClient uses the same discovery logic as the Producer                                          
   34     admin_client = AdminClient({'bootstrap.servers': BOOTSTRAP_SERVERS})                                     
   35                                                    

In [4]:
import os
import signal
import multiprocessing

from producer import run_producer

process = multiprocessing.Process(target=run_producer)
process.start()

def kill_producer():
    os.kill(process.pid, signal.SIGINT)
    process.join()

🔍 Connecting to Bootstrap Server: localhost:9092...



✅ Cluster Discovery Successful!
   Cluster ID: 5L6g3nShT-eMCtK--X86sw
   Controller Broker ID: 1
   ------------------------------------------------
   Discovered Brokers (1 found):
   - Broker ID: 1 | Address: localhost:9092
   ------------------------------------------------

⚠️ Topic 'words-topic-spark' not found. Creating...
✅ Topic 'words-topic-spark' created
🚀 Starting Producer to topic: words-topic-spark
Press Ctrl+C to stop.
✅ Message delivered to words-topic-spark [0] offset 0
✅ Message delivered to words-topic-spark [1] offset 0
✅ Message delivered to words-topic-spark [3] offset 0
✅ Message delivered to words-topic-spark [1] offset 1
✅ Message delivered to words-topic-spark [3] offset 1
✅ Message delivered to words-topic-spark [0] offset 1
✅ Message delivered to words-topic-spark [0] offset 2
✅ Message delivered to words-topic-spark [1] offset 2
✅ Message delivered to words-topic-spark [0] offset 3
✅ Message delivered to words-topic-spark [1] offset 3
✅ Message delivered to

To connect to a Kafka topic, you must specify the broker address and the topic we want to listen to:

In [5]:
kafka_stream = (
    spark
        .readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "localhost:9092")
        .option("subscribe", "words-topic-spark")
        # Read from the beginning if data hasn't been seen yet
        .option("startingOffsets", "earliest")
        # Read no more than 10k messages at a time
        .option("maxOffsetsPerTrigger", 10000) 
        # If Kafka deleted old messages (due to retention policy),
        #   and Spark hasn't managed to read them yet, by default Spark will crash with an error.
        # This option forces it to simply skip deleted data and continue working.
        # In production, setting this to 'false' effectively disables 'At-Least-Once' guarantees if data is expired in Kafka. 
        # Only use if partial data loss is acceptable.
        .option("failOnDataLoss", "false")
        .load()
)

Let's implement a simple example with Word Count:

In [6]:
kafka_words_df = (
    kafka_stream
        .withWatermark("timestamp", "10 seconds")
        .select(F.col("value").cast("string").alias("sentence"))
        .select(F.explode(F.split(F.col("sentence"), " ")).alias("word"))
        .groupBy("word")
        .count()
)

For optimized saving to a Delta Table with EO guarantee, implement custom processing for each batch:

In [7]:
from delta.tables import DeltaTable

def process_batch(batch_df, batch_id, table_path="./artifacts/delta_table"):
    batch_df.persist()
    
    # Check if Delta table already exists
    if not DeltaTable.isDeltaTable(spark, table_path):
        # If table doesn't exist (first run), just write data
        print("Initializing Delta Table...")
        (
            batch_df
                .write
                .format("delta")
                .mode("overwrite")
                .save(table_path)
        )
    else:
        # B. If table exists, perform MERGE
        delta_table = DeltaTable.forPath(spark, table_path)
        
        # MERGE Syntax:
        # target - is the Delta table on disk
        # source - is our current micro-batch
        (delta_table.alias("target")
            .merge(
                source=batch_df.alias("source"),
                # Deduplication key
                condition="target.word = source.word" 
            )
            .whenMatchedUpdate(set={
                # IMPORTANT: In 'update' mode streaming sends the FULL updated sum.
                # Therefore, we simply overwrite the value, not add to it.
                # If outputMode was 'append', we would do target.count + source.count
                "count": "source.count" 
            })
            .whenNotMatchedInsert(values={
                "word": "source.word",
                "count": "source.count"
            })
            .execute()
        )

        # Optimize table after every 5 batches
        # In high-load systems, OPTIMIZE is moved to a separate process, 
        #   so as not to block the reading stream.
        if batch_id % 5 == 0:
            print(f"Optimizing Delta Table after the batch {batch_id}...")
            delta_table.optimize().executeCompaction()
        
    batch_df.unpersist()

```bash
./kafka-topics.sh --bootstrap-server localhost:9092 --create --topic words-topic-spark
./kafka-console-producer.sh --bootstrap-server localhost:9092 --topic words-topic-spark
```

In [8]:
kafka_query = (
    kafka_words_df.writeStream
        .trigger(processingTime="5 seconds")
        .queryName('kafka_words')
        .outputMode("update")
        .option("checkpointLocation", "artifacts/checkpoints/kafka_words")
        .foreachBatch(process_batch)
        .start()
)
kafka_query.awaitTermination(timeout=60)
kafka_query.stop()

25/12/10 22:47:39 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
25/12/10 22:47:40 WARN AdminClientConfig: The configuration 'key.deserializer' was supplied but isn't a known config.
25/12/10 22:47:40 WARN AdminClientConfig: The configuration 'value.deserializer' was supplied but isn't a known config.
25/12/10 22:47:40 WARN AdminClientConfig: The configuration 'enable.auto.commit' was supplied but isn't a known config.
25/12/10 22:47:40 WARN AdminClientConfig: The configuration 'max.poll.records' was supplied but isn't a known config.
25/12/10 22:47:40 WARN AdminClientConfig: The configuration 'auto.offset.reset' was supplied but isn't a known config.


Initializing Delta Table...


25/12/10 22:47:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/12/10 22:47:58 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 18035 milliseconds
25/12/10 22:48:08 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 10312 milliseconds
25/12/10 22:48:13 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 5663 milliseconds
25/12/10 22:48:19 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 5376 milliseconds
25/12/10 22:48:25 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 5793 milliseconds


Optimizing Delta Table after the batch 5...


25/12/10 22:48:31 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 6770 milliseconds
25/12/10 22:48:39 WARN BlockManager: Block rdd_506_0 could not be removed as it was not found on disk or in memory
25/12/10 22:48:39 WARN BlockManager: Block rdd_506_14 could not be removed as it was not found on disk or in memory
25/12/10 22:48:39 WARN BlockManager: Block rdd_506_15 could not be removed as it was not found on disk or in memory
25/12/10 22:48:39 WARN BlockManager: Block rdd_506_2 could not be removed as it was not found on disk or in memory
25/12/10 22:48:39 WARN BlockManager: Block rdd_506_5 could not be removed as it was not found on disk or in memory
25/12/10 22:48:39 WARN BlockManager: Block rdd_506_8 could not be removed as it was not found on disk or in memory
25/12/10 22:48:39 WARN BlockManager: Block rdd_506_13 could not be removed as it was not found on disk or in memory
25/12/10 22:48:39 WARN BlockManager: Bloc

In [9]:
kill_producer()

Let's see what happened:

In [10]:
(
    spark
        .read
        .format("delta")
        .load('./artifacts/delta_table')
        .show()
)

+------------+-----+
|        word|count|
+------------+-----+
|        acid|   19|
|         and|   32|
|      apache|   14|
|architecture|   14|
|       batch|   14|
|         big|   11|
|   computing|   11|
|        data|   52|
|       delta|   19|
| distributed|   11|
| engineering|   14|
|     ensures|   19|
|         for|   14|
|         fun|   18|
|       great|   14|
|      hadoop|   18|
|     handles|   14|
|          is|   64|
|       kafka|   13|
|        lake|   19|
+------------+-----+
only showing top 20 rows

